In [ ]:
#| default_exp core

# Core

> Authentication, compliance profiles, label helpers, and `GenAIStack` orchestrator.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os
from fastcore.basics import store_attr

try:
    import google.auth
    import google.auth.transport.requests
    import google.oauth2.service_account as sa_creds
    from google.cloud import resourcemanager_v3
except ImportError:
    pass  # optional at import time; raised at runtime if needed

In [ ]:
#| export
HIPAA = dict(
    encryption=True,
    tls_min='1.2',
    audit=True,
    multi_region=True,
    backup_retention=35,
    deletion_protection=True,
    labels={'compliance': 'hipaa'},
)

ISO27001 = dict(
    encryption=True,
    audit=True,
    managed_sa=True,
    least_privilege=True,
    tls_min='1.2',
    labels={'compliance': 'iso27001'},
)

SOC2 = dict(
    encryption=True,
    audit=True,
    mfa_required=True,
    backup_retention=7,
    labels={'compliance': 'soc2'},
)

In [ ]:
#| export
class GCPAuth:
    """Application Default Credentials wrapper for GCP. Reads `GOOGLE_CLOUD_PROJECT`
    and `GOOGLE_CLOUD_REGION` from env. Pass `service_account_file` for key-based auth."""

    def __init__(self, project=None, region=None, service_account_file=None,
                 impersonate_sa=None):
        store_attr()
        self.project = project or os.environ.get('GOOGLE_CLOUD_PROJECT') or os.environ.get('GCLOUD_PROJECT')
        self.region  = region  or os.environ.get('GOOGLE_CLOUD_REGION', 'us-central1')
        if not self.project:
            raise ValueError('project required; set GOOGLE_CLOUD_PROJECT or pass project=')

        if service_account_file:
            self.credentials = sa_creds.Credentials.from_service_account_file(
                service_account_file,
                scopes=['https://www.googleapis.com/auth/cloud-platform'],
            )
        else:
            self.credentials, _ = google.auth.default(
                scopes=['https://www.googleapis.com/auth/cloud-platform']
            )

        if impersonate_sa:
            from google.auth import impersonated_credentials
            self.credentials = impersonated_credentials.Credentials(
                source_credentials=self.credentials,
                target_principal=impersonate_sa,
                target_scopes=['https://www.googleapis.com/auth/cloud-platform'],
            )

    def __repr__(self):
        return f'GCPAuth(project={self.project!r}, region={self.region!r})'

In [ ]:
#| export
def label_resources(auth, labels: dict) -> list:
    """List GCP project resources matching `labels` using Cloud Asset Inventory."""
    from google.cloud import asset_v1
    client = asset_v1.AssetServiceClient(credentials=auth.credentials)
    label_expr = ' AND '.join(f'labels.{k}={v}' for k, v in labels.items())
    req = asset_v1.SearchAllResourcesRequest(
        scope=f'projects/{auth.project}',
        query=label_expr,
    )
    return list(client.search_all_resources(request=req))


def list_labeled_resources(auth) -> list:
    """List all labeled resources in the project."""
    return label_resources(auth, {})

In [ ]:
#| export
class GenAIStack:
    """Provision a full enterprise GenAI stack on GCP in one call."""

    def __init__(self, auth: GCPAuth, name: str, compliance: dict = None):
        store_attr()
        self._resources = {}
        self._compliance = compliance or {}

    def provision(
        self,
        vertex_ai: bool = True,
        vector_search: bool = False,
        gcs: bool = True,
        firestore: bool = True,
        cloud_sql: bool = False,
        memorystore: bool = True,
        gke: bool = False,
        secret_manager: bool = True,
    ) -> dict:
        """Provision selected GCP services. Returns dict of resource identifiers."""
        from gcpeasy.network import create_service_account, bind_iam_role, create_secret
        from gcpeasy.data import create_bucket, create_collection, create_redis
        from gcpeasy.ai import create_vector_search_index

        name = self.name
        co = self._compliance
        labels = co.get('labels', {})
        labels.update({'gcpeasy': name})

        # Service Account
        sa = create_service_account(self.auth, f'{name}-sa',
                                    display_name=f'{name} GenAI SA')
        bind_iam_role(self.auth, sa['email'],
                      'roles/aiplatform.user')
        self._resources['service_account'] = sa['email']

        if gcs:
            bucket = create_bucket(self.auth, f'{name}-data', labels=labels, **co)
            self._resources['gcs_bucket'] = bucket['name']

        if firestore:
            coll = create_collection(self.auth, name)
            self._resources['firestore_collection'] = coll

        if memorystore:
            redis = create_redis(self.auth, f'{name}-cache', **co)
            self._resources['memorystore'] = redis.get('name')

        if vector_search:
            idx = create_vector_search_index(self.auth, f'{name}-index', dimensions=768)
            self._resources['vector_search_index'] = idx.get('name')

        if secret_manager:
            sec = create_secret(self.auth, f'{name}/api-key', 'placeholder',
                                labels=labels)
            self._resources['secret'] = sec.get('name')

        return self._resources

    def summary(self) -> dict:
        "Return provisioned resource identifiers."
        return self._resources